In [0]:
catalog_name = dbutils.widgets.get('catalog_name')
spark.sql(F'USE CATALOG {catalog_name}')


In [0]:
%sql

CREATE SCHEMA IF NOT EXISTS bronze;

###SQL Data Ingestion

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

CUSTOMERS_SCHEMA = StructType(
    [
        StructField("customer_id", IntegerType()),
        StructField("first_name", StringType()),
        StructField("last_name", StringType()),
        StructField("email", StringType()),
        StructField("phone", StringType()),
        StructField("city", StringType()),
        StructField("state", StringType()),
        StructField("country", StringType()),
        StructField("customer_status", StringType()),
        StructField("created_at", TimestampType()),
        StructField("updated_at", TimestampType()),
    ]
)
 
ORDERS_SCHEMA = StructType(
    [
        StructField("order_id", IntegerType()),
        StructField("customer_id", DecimalType()),
        StructField("order_date", TimestampType()),
        StructField("order_status", StringType()),
        StructField("shipping_address", StringType()),
        StructField("total_amount", DecimalType()),
        StructField("created_at", TimestampType()),
        StructField("updated_at", TimestampType()),
    ]
)

ORDER_ITEMS_SCHEMA = StructType(
    [
        StructField("order_item_id", LongType()),
        StructField("order_id", LongType()),
        StructField("product_id", IntegerType()),
        StructField("quantity",IntegerType()),
        StructField("unit_price", DecimalType()),
        StructField("discount", DecimalType()),
        StructField("created_at", TimestampType()),
        StructField("updated_at", TimestampType()),
    ]
)

PAYMENTS_SCHEMA = StructType(
    [
        StructField("payment_id", LongType()),
        StructField("order_id", LongType()),
        StructField("payment_method", StringType()),
        StructField("payment_amount", DecimalType()),
        StructField("payment_status", StringType()),
        StructField("payment_date", TimestampType()),
        StructField("updated_at", TimestampType()),
    ]
)




In [0]:
customers_df = (spark.readStream
    .format("csv")
    .option("header", "true")
    .schema(CUSTOMERS_SCHEMA)
    .load("/Volumes/shopsphere/ecomm_schema/capstone_data/sql_server/initial/customers/"))
 
customers_df.writeStream.option(
    "checkpointLocation",
    "/Volumes/shopsphere/ecomm_schema/capstone_data/checkpoint/customers",
).format('delta').trigger(availableNow=True).outputMode("Append").toTable(
    "shopsphere.bronze.customers"
)

In [0]:
orders_df = (spark.readStream
             .format("csv")
             .option("header", "true")
             .schema(ORDERS_SCHEMA)
             .load("/Volumes/shopsphere/ecomm_schema/capstone_data/sql_server/initial/orders/"))

orders_df = orders_df.withColumn("customer_id", col("customer_id").cast("integer"))
 
orders_df.writeStream.option(
    "checkpointLocation",
    "/Volumes/shopsphere/ecomm_schema/capstone_data/checkpoint/orders",
).format('delta').trigger(availableNow=True).outputMode("Append").toTable(
    "shopsphere.bronze.orders"
)

In [0]:
order_items_df = (spark.readStream
             .format("csv")
             .option("header", "true")
             .schema(ORDER_ITEMS_SCHEMA)
             .load("/Volumes/shopsphere/ecomm_schema/capstone_data/sql_server/initial/order_items/"))


 
order_items_df.writeStream.option(
    "checkpointLocation",
    "/Volumes/shopsphere/ecomm_schema/capstone_data/checkpoint/order_items",
).format('delta').trigger(availableNow=True).outputMode("Append").toTable(
    "shopsphere.bronze.order_items"
)

In [0]:
payments_df = (spark.readStream
             .format("csv")
             .option("header", "true")
             .schema(PAYMENTS_SCHEMA)
             .load("/Volumes/shopsphere/ecomm_schema/capstone_data/sql_server/initial/payments/"))


 
payments_df.writeStream.option(
    "checkpointLocation",
    "/Volumes/shopsphere/ecomm_schema/capstone_data/checkpoint/payments",
).format('delta').trigger(availableNow=True).outputMode("Append").toTable(
    "shopsphere.bronze.payments"
)

In [0]:

from delta.tables import DeltaTable

delta_table = DeltaTable.forName(spark, "shopsphere.bronze.orders")

delta_table.history().display()